## **Step 1:** Uploading the Dataset

In [1]:
from google.colab import files
uploaded = files.upload()

Saving marketing_campaign.csv to marketing_campaign (1).csv


##**Step 2:** Import Libraries

In [2]:
import pandas as pd
import numpy as np

# Purpose: pandas used for data handling and numpy is used for numeric operations used in outlier calculations.

##**Step 3**: Loading the Dataset

In [3]:
df = pd.read_csv('/content/marketing_campaign.csv', sep='\t')
df.head()

# Purpose: This file is tab-separated, not comma-separated. sep='\t' is required without it, all 29 columns will collapse into one, and nothing else will work.

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


**Initial Load:** Dataset loaded with 2,240 rows and 29 columns of customer demographic and purchasing data.

##**Step 4:** Data Quality Report

In [4]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum()>0])
print("\nDuplicate rows:", df.duplicated().sum())

# Purpose: Establishes "before" snapshot, filtering nulls to only show affected columns keeps output readable.

Shape: (2240, 29)

Data types:
 ID                       int64
Year_Birth               int64
Education               object
Marital_Status          object
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer             object
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Z_CostContact            int64
Z_Revenue                int64
Response                 int64
dtype: object

Missing values:
 Income    24
dtype: int64

Duplicate 

**Data Quality Report:** 2,240 rows, 29 columns. Income has 24 missing values, no other column has nulls. No duplicate rows found.

##**Step 5:** Inspect Categorical Columns

In [5]:
print("Marital_Status values:", df['Marital_Status'].unique())
print("\nEducation values:", df['Education'].unique())

# Purpose:  .unique() reveals every distinct category — this is how messy labels get caught.

Marital_Status values: ['Single' 'Together' 'Married' 'Divorced' 'Widow' 'Alone' 'Absurd' 'YOLO']

Education values: ['Graduation' 'PhD' 'Master' 'Basic' '2n Cycle']


**Category Inspection:** Marital_Status contains "Absurd" and "YOLO" — clearly not real categories, likely joke/error entries. "Alone" overlaps in meaning with "Single." Education values are all valid (including "2n Cycle," a legitimate European qualification level).

##**Step 6:** Standardize Marital_Status

In [6]:
df['Marital_Status'] = df['Marital_Status'].replace({
    'Alone': 'Single',
    'Absurd': 'Single',
    'YOLO': 'Single'
})
print(df['Marital_Status'].value_counts())

# Purpose: Maps problematic values to a sensible category rather than deleting those customers, since their other data is still valid.

Marital_Status
Married     864
Together    580
Single      487
Divorced    232
Widow        77
Name: count, dtype: int64


**Decision: Category Standardization:** Merged "Alone" into "Single." Reclassified "Absurd" and "YOLO" as "Single" since only this one field was corrupted, all other customer data remains usable.

##**Step 7:** Handling Missing Income

In [7]:
print("Income summary before cleaning:\n", df['Income'].describe())
median_income = df['Income'].median()
df['Income'] = df['Income'].fillna(median_income)
print("\nMissing values remaining:", df['Income'].isnull().sum())

# Purpose: Median is used instead of mean because income is typically right-skewed by high earners, media better represents a "typical" customer.

Income summary before cleaning:
 count      2216.000000
mean      52247.251354
std       25173.076661
min        1730.000000
25%       35303.000000
50%       51381.500000
75%       68522.000000
max      666666.000000
Name: Income, dtype: float64

Missing values remaining: 0


**Decision: Missing Value Handle**
24 missing Income values filled with median(~$51,382), chosen over the mean due to income's natural skew.

##**Step 8:** Fix Dt_Customer Data Type

In [8]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
print(df['Dt_Customer'].dtype)

# Purpose: Converts the join-date column from text to real datetime, enabling date-based analysis and satisfying the "correct dtype" requirement.

datetime64[ns]


**Data Type Correction:** Converted Dt_Customer from text to proper datetime format.

##**Step 9:** Outlier Detection: Year_Birth

In [9]:
print("Year_Birth range:", df['Year_Birth'].min(), "to", df['Year_Birth'].max())
suspicious_birth = df[df['Year_Birth']<1940]
print(f"\nSuspicious birth years found: {len(suspicious_birth)}")
print(suspicious_birth[['ID', 'Year_Birth']])

# Purpose: The earliest Year_Birth is 1893; implying an age over 130, clearly a data error. A business-logic check works better here than a pure statistical formula.

Year_Birth range: 1893 to 1996

Suspicious birth years found: 3
        ID  Year_Birth
192   7829        1900
239  11004        1893
339   1150        1899


**Outlier Detection:** Found customers with Year_Birth as early as 1893, implausibles ages, clear data entry errors.

##**Step 10:** Remove Year_Birth Outliers

In [10]:
df = df[df['Year_Birth']>=1940]
print("Rows remaining:", df.shape[0])

# Purpose: Since this affects only a handful of rows, removal is safer than trying to fabricate a replacement birth year.

Rows remaining: 2237


**Decision: Outlier Removal**
Rows with birth years before 1940 were removed, a small number of clear errors, not valid extreme values.

##**Step 11:** Outlier Detection: Income (IQR Method)

In [11]:
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
income_outliers = df[(df['Income'] < lower_bound) | (df['Income'] > upper_bound)]
print(f"\nIncome outliers found: {len(income_outliers)}")
print(income_outliers[['ID', 'Income']])

# Purpose: IQR is the standard statistical method, anything beyond 1.5× the interquartile range is flagged. This catches the extreme $666,666 entry.


Income outliers found: 8
         ID    Income
164    8475  157243.0
617    1503  162397.0
655    5555  153924.0
687    1501  160803.0
1300   5336  157733.0
1653   4931  157146.0
2132  11181  156924.0
2233   9432  666666.0


**Outlier Detection:** Income
IQR method identified outliers including one extreme value of $666,666, far beyond the typical range.

##**Step 12:** Cap Income Outliers

In [12]:
df['Income'] = np.where(df['Income'] > upper_bound, upper_bound, df['Income'])
print("Income summary after capping:\n", df['Income'].describe())

# Purpose: Capping preserves these customers' other valuable data (spending, purchases) while preventing the extreme value from distorting analysis, chosen over deletion.

Income summary after capping:
 count      2237.000000
mean      51854.814037
std       20936.241476
min        1730.000000
25%       35523.000000
50%       51381.500000
75%       68281.000000
max      117418.000000
Name: Income, dtype: float64


**Decision: Capping vs. Removal**
Income outliers were capped, not removed, to retain valuable behavioral data from these customers.

##**Step 13:** Before vs. After Summary Table

In [13]:
summary = pd.DataFrame({
    'Metric': ['Total Rows', 'Null Values (Income)', 'Duplicate Rows', 'Dt_Customer dtype'],
    'Before Cleaning': [2240, 24, 0, 'object (text)'],
    'After Cleaning': [df.shape[0], df['Income'].isnull().sum(), df.duplicated().sum(), str(df['Dt_Customer'].dtype)]
})
summary

# Purpose: Direct before/after comparison required by the checklist.

,Metric,Before Cleaning,After Cleaning
0,Total Rows,2240,2237
1,Null Values (Income),24,0
2,Duplicate Rows,0,0
3,Dt_Customer dtype,object (text),datetime64[ns]


##**Step 14:** Saving Cleaned Dataset

In [14]:
df.to_csv('marketing_campaign_cleaned.csv', index=False)
files.download('marketing_campaign_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##**Step 15:** Conclusion

## **Conclusion**
This dataset required moderate cleaning: standardizing 3 inconsistent Marital_Status entries, imputing 24 missing Income values via median, correcting Dt_Customer's data type, and handling two types of outliers, removing implausible birth years and capping extreme income. The cleaned dataset is now ready for customer segmentation analysis.